<a href="https://colab.research.google.com/github/dakshayani2906/Data-Science-Portfolio/blob/main/Shikasen_Trave_Experience.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

In [33]:
train_travel = pd.read_csv("Traveldata_train_(1).csv")
train_survey = pd.read_csv("Surveydata_train_(1).csv")
test_travel  = pd.read_csv("Traveldata_test_(1).csv")
test_survey  = pd.read_csv("Surveydata_test_(1).csv")

In [34]:
train_df = train_travel.merge(train_survey, on="ID", how="inner", validate="one_to_one")
test_df  = test_travel.merge(test_survey, on="ID", how="inner", validate="one_to_one")

In [35]:
target_col = "Overall_Experience"
y = train_df[target_col].astype(int)
X = train_df.drop(columns=[target_col])
X_test = test_df.copy()

In [36]:
ordinal_map = {'Poor':0, 'Needs Improvement':1, 'Acceptable':2, 'Good':3, 'Excellent':4}

ordinal_cols = [
    'Seat_Comfort','Food_and_Drink','Entrance_and_Exit','Online_Support',
    'Onboard_Wifi_Service','Onboard_Entertainment','Turnaround_Time','Online_Boarding',
    'Legroom','Baggage_Handling','CheckIn_Service','Cleanliness','Onboard_Service',
    'Luggage_Handling','Inflight_wifi_service','Inflight_entertainment','Gate_location',
    'Ease_of_Online_Booking','Customer_Service'
]

In [37]:
ordinal_cols = [c for c in ordinal_cols if c in X.columns]

nominal_cols = [c for c in ['Gender','Customer_Type','Type_Travel','Travel_Class'] if c in X.columns]
numeric_cols = [c for c in ['Age','Travel_Distance','Departure_Delay_in_Mins','Arrival_Delay_in_Mins'] if c in X.columns]


In [38]:
def apply_ordinal(df):
    for c in ordinal_cols:
        df[c] = df[c].map(ordinal_map).astype('float')  # float to allow median impute
    return df

X = apply_ordinal(X.copy())
X_test = apply_ordinal(X_test.copy())

In [39]:
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

In [40]:
cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

pre = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numeric_cols),
        ("nom", cat_pipe, nominal_cols),
        # Ordinals have already been mapped to numeric; include them in numeric branch too:
        ("ord", SimpleImputer(strategy="median"), ordinal_cols),
    ],
    remainder="drop"
)

In [41]:
logit = Pipeline(steps=[
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=500, n_jobs=1, solver="saga", random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(logit, X, y, scoring="accuracy", cv=cv, n_jobs=1)
print("LogReg CV Accuracy: ", cv_scores.mean().round(4), "+/-", cv_scores.std().round(4))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogReg CV Accuracy:  0.7532 +/- 0.0031


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [42]:
hgb = Pipeline(steps=[
    ("pre", pre),
    ("clf", HistGradientBoostingClassifier(
        max_depth=None, learning_rate=0.08, max_iter=400, random_state=42
    ))
])

cv_scores_hgb = cross_val_score(hgb, X, y, scoring="accuracy", cv=cv, n_jobs=1)
print("HGB CV Accuracy: ", cv_scores_hgb.mean().round(4), "+/-", cv_scores_hgb.std().round(4))

HGB CV Accuracy:  0.9412 +/- 0.0018


In [43]:
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
hgb.fit(X_tr, y_tr)
va_pred = hgb.predict(X_va)
print("HGB Hold-out Accuracy:", round(accuracy_score(y_va, va_pred), 4))

HGB Hold-out Accuracy: 0.9418


In [48]:
fitted_pre = hgb.named_steps['pre']  # this is fitted on X_tr,y_tr

# Build feature names in the same order ColumnTransformer outputs them
feat_names = []

# 1) numeric block
feat_names += [*numeric_cols]

# 2) nominal block (OHE)
if 'nom' in fitted_pre.named_transformers_ and nominal_cols:
    ohe = fitted_pre.named_transformers_['nom'].named_steps['ohe']
    ohe_names = ohe.get_feature_names_out(nominal_cols)
    feat_names += list(ohe_names)

# 3) ordinal block (imputed numeric)
feat_names += [*ordinal_cols]

# Sanity check (optional): number of transformed features vs names
from sklearn.utils._estimator_html_repr import _write_label_html  # not needed; just a trick to force transform
Xt_va = fitted_pre.transform(X_va)
print("Transformed features:", Xt_va.shape[1], " | Names:", len(feat_names))

Transformed features: 23  | Names: 23


In [55]:
hgb.fit(X, y)
test_pred = hgb.predict(X_test)

In [59]:

submission = pd.DataFrame({"ID": X_test["ID"], "Overall_Experience": test_pred})
# Safety: if your platform requires exactly 35,602 rows:
print("Submission shape:", submission.shape)
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")

Submission shape: (35602, 2)
Saved submission.csv


In [60]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>